In [1]:
from google.colab import drive
drive.mount('/content/drive/')


ValueError: mount failed

In [ ]:
!ls /content/drive/


In [ ]:
from google.colab import files
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import pickle
import os
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import LabelEncoder

import warnings
warnings.filterwarnings('ignore')

## Dataset

In [ ]:
# uploaded = files.upload()
# filename = list(uploaded.keys())[0]
# print(f"Fichier '{filename}' uploade")
df = pd.read_csv('/content/drive/MyDrive/python/WA_Fn-UseC_-HR-Employee-Attrition.csv')
print("Dataset loaded!")

In [ ]:
# df = pd.read_csv(filename)
df.head()


In [ ]:
df.shape

In [ ]:
df.info()


In [ ]:
df.describe(include="all")


In [ ]:
df.describe()


# single unique value

In [ ]:
single_value_cols = df.columns[df.nunique() == 1]

print(single_value_cols.tolist())

In [ ]:
df.drop(columns=['EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours'], inplace=True)


In [ ]:
business_features = [
    'JobLevel',
    'JobSatisfaction',
    'EnvironmentSatisfaction',
    'RelationshipSatisfaction',
    'PercentSalaryHike',
    'StockOptionLevel',
    'MonthlyIncome',
    'MonthlyRate',
    'HourlyRate',
    'DailyRate',
    'YearsAtCompany',
    'YearsInCurrentRole',
    'YearsSinceLastPromotion',
    'YearsWithCurrManager',
    'NumCompaniesWorked',
    'OverTime',
    'JobRole',
    'Department',
    'WorkLifeBalance',
    'EducationField',
]

df = df.drop(columns=business_features, errors='ignore')


In [ ]:
df.head()

In [ ]:
print("Remaining features:", df.shape[1])


In [ ]:
df.shape

# columns classification

In [ ]:
target = ['Attrition']

quantitative = [
    'Age',
    'DistanceFromHome',
    'TotalWorkingYears',
    'TrainingTimesLastYear'
]

ordinal = [
    'Education',
]

nominal = [
    'BusinessTravel',
    'Gender',
    'MaritalStatus',
]


In [ ]:
set(df.columns) == set(target + quantitative + ordinal + nominal)

# EDA

target


In [ ]:
sns.countplot(x='Attrition', data=df)
plt.title('Attrition Distribution')
plt.show()

df['Attrition'].value_counts(normalize=True) * 100

nominal

In [ ]:

cols = nominal
n = len(cols)
ncols = 4
nrows = (n + ncols - 1) // ncols

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols*5, nrows*5))

axes = axes.flatten()

for i, col in enumerate(cols):
    df[col].value_counts().plot.pie(
        ax=axes[i],
        autopct='%1.1f%%',
        startangle=90,
        title=col
    )
    axes[i].set_ylabel("")

# Remove empty subplots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


ordinal

In [ ]:
for col in ordinal:
    plt.figure(figsize=(6,4))
    sns.countplot(x=col, data=df, order=df[col].unique())
    plt.title(f"Count of {col}")
    plt.show()

In [ ]:
sns.set(style="whitegrid")

plt.figure(figsize=(8,5))
sns.countplot(x='Education', hue='Attrition', data=df, palette='Set2')

plt.title("Attrition by Education ")
plt.xlabel("Education")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.legend(title='Attrition', labels=['No', 'Yes'])
plt.tight_layout()
plt.show()


In [ ]:
for col in ordinal:
    counts = df[col].value_counts().sort_index()

    plt.figure(figsize=(6,6))
    plt.pie(
        counts,
        labels=counts.index,
        autopct='%1.1f%%',
        startangle=90
    )
    plt.title(f"Distribution of {col} (ordered)")
    plt.show()


Distribution plots

In [ ]:
n_cols = 4
n_rows = math.ceil(len(quantitative)/n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(quantitative):
    sns.histplot(df[col], kde=True, ax=axes[i])
    axes[i].set_title(col)

# Removing empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


Correlation

In [ ]:
corr = df[quantitative].corr()

plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Matrix of Quantitative Features")
plt.show()


Attrition vs features

In [ ]:
# Continuous features
n_cols = 4
n_rows = math.ceil(len(quantitative)/n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(quantitative):
    sns.boxplot(x='Attrition', y=col, data=df, ax=axes[i])
    axes[i].set_title(f"{col} vs Attrition")

# Removing empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


In [ ]:
# Categorical features
n_cols = 4
n_rows = math.ceil(len(nominal)/n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()

for i, col in enumerate(nominal):
    sns.countplot(x=col, hue='Attrition', data=df, ax=axes[i])
    axes[i].set_title(f"{col} vs Attrition")
    axes[i].tick_params(axis='x', rotation=45)

# Removing empty subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()


Check skewed / imbalanced features

In [ ]:
df[quantitative].skew().sort_values(ascending=False)


pairplot / scatter matrix

In [ ]:
sns.pairplot(df[quantitative + ['Attrition']], hue='Attrition')
plt.show()


# missing values

In [ ]:
(df.isnull().mean() * 100)


# outliers

In [ ]:
n_cols = 4
n_rows = math.ceil(len(quantitative) / n_cols)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows))
axes = axes.flatten()
df_before = df.copy()

for i, col in enumerate(quantitative):
    sns.boxplot(x=df_before[col], ax=axes[i])
    axes[i].set_title(f"{col} (before)")

for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])


plt.tight_layout()
plt.show()

In [ ]:
outlier_percentages = {}

for col in quantitative:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    percentage = (len(outliers) / len(df)) * 100

    outlier_percentages[col] = percentage


outlier_df = pd.DataFrame.from_dict(
    outlier_percentages,
    orient='index',
    columns=['Outlier Percentage (%)']
)

outlier_df = outlier_df.sort_values(by='Outlier Percentage (%)', ascending=False)
outlier_df


# Encoding

In [ ]:
df.columns

Target

In [ ]:
df['Attrition'] = df['Attrition'].map({'No': 0, 'Yes': 1})

nominal

In [ ]:
label_encoders = {}

for col in nominal:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])

    # save mapping
    label_encoders[col] = dict(zip(le.classes_, le.transform(le.classes_)))


In [ ]:
clean_label_encoders = {
    col: {int(k) if isinstance(k, (np.integer, int)) else k: int(v) for k, v in mapping.items()}
    for col, mapping in label_encoders.items()
}

In [ ]:
save_path = '/content/drive/MyDrive/python/label_encoders.pkl'
os.makedirs(os.path.dirname(save_path), exist_ok=True)

# Save the dictionary
with open(save_path, 'wb') as f:
    pickle.dump(clean_label_encoders, f)

print(f"Label encoders saved to {save_path}")

In [ ]:
with open(save_path, 'rb') as f:
    loaded_encoders = pickle.load(f)

print(loaded_encoders)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
# df.to_csv("cleanedData.csv", index=False)
# files.download("cleanedData.csv")

df.to_csv('/content/drive/MyDrive/python/cleanedDataset.csv', index=False)
print(f"Dataset saved to: /content/drive/MyDrive/python")